# ABA — Adult BMI Assessment
**HEDIS 2024 | Measurement year: 2021 (Jan 1 – Dec 31)**

**Denominator:** Members 18–74 as of Dec 31, 2021 with at least one qualifying outpatient visit during the measurement year.

**Numerator:** Denominator members with a BMI value documented during the measurement year.

**Direction:** Higher is better.

---

**Claims-based approach:**
BMI documentation is identified via ICD-10 Z68.x codes on carrier and outpatient claims.
HEDIS ABA also requires a follow-up plan for members with BMI outside the normal range (< 18.5 or ≥ 25).
Follow-up plan documentation is an EHR element and cannot be verified from claims.
This implementation uses Z68.x code presence as the numerator criteria.

**Qualifying visit codes:**
Preventive E&M (99385–99387, 99395–99397) and office/outpatient E&M (99202–99205, 99212–99215).
CPT 99201 excluded — retired effective Jan 1, 2021.

## Connection

In [1]:
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
import getpass
from urllib.parse import quote_plus

password = getpass.getpass('SA password: ')

conn_str = (
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=localhost;'
    'DATABASE=hedis;'
    'UID=SA;'
    f'PWD={password};'
    'TrustServerCertificate=yes;'
)

engine = create_engine(f'mssql+pyodbc:///?odbc_connect={quote_plus(conn_str)}')
conn = engine.connect()
print('Connected.')

SA password:  ········


Connected.


## Step 1 — Denominator
Members 18–74 as of Dec 31, 2021, alive, with at least one qualifying outpatient visit in 2021.

In [2]:
pd.read_sql("""
    WITH age_eligible AS (
        SELECT BENE_ID
        FROM beneficiary
        WHERE BENE_DEATH_DT IS NULL
          AND FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25)
              BETWEEN 18 AND 74
    ),
    qualifying_visits AS (
        SELECT DISTINCT BENE_ID
        FROM (
            SELECT BENE_ID, HCPCS_CD, CLM_FROM_DT FROM carrier
            UNION ALL
            SELECT BENE_ID, HCPCS_CD, CLM_FROM_DT FROM outpatient
        ) h
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
          AND HCPCS_CD IN (
              '99385', '99386', '99387',
              '99395', '99396', '99397',
              '99202', '99203', '99204', '99205',
              '99212', '99213', '99214', '99215'
          )
    )
    SELECT COUNT(DISTINCT a.BENE_ID) AS denominator
    FROM age_eligible a
    INNER JOIN qualifying_visits v ON a.BENE_ID = v.BENE_ID
""", conn)

,denominator
0,341


## Step 2 — Numerator
Denominator members with a BMI ICD-10 code (Z68.x) on any claim in 2021.

In [3]:
pd.read_sql("""
    WITH age_eligible AS (
        SELECT BENE_ID
        FROM beneficiary
        WHERE BENE_DEATH_DT IS NULL
          AND FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25)
              BETWEEN 18 AND 74
    ),
    qualifying_visits AS (
        SELECT DISTINCT BENE_ID
        FROM (
            SELECT BENE_ID, HCPCS_CD, CLM_FROM_DT FROM carrier
            UNION ALL
            SELECT BENE_ID, HCPCS_CD, CLM_FROM_DT FROM outpatient
        ) h
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
          AND HCPCS_CD IN (
              '99385', '99386', '99387',
              '99395', '99396', '99397',
              '99202', '99203', '99204', '99205',
              '99212', '99213', '99214', '99215'
          )
    ),
    denominator AS (
        SELECT a.BENE_ID
        FROM age_eligible a
        INNER JOIN qualifying_visits v ON a.BENE_ID = v.BENE_ID
    ),
    numerator AS (
        SELECT DISTINCT d.BENE_ID
        FROM denominator d
        INNER JOIN (
            SELECT BENE_ID, LINE_ICD_DGNS_CD AS dx, CLM_FROM_DT FROM carrier
            UNION ALL
            SELECT BENE_ID, PRNCPAL_DGNS_CD,         CLM_FROM_DT FROM outpatient
            UNION ALL
            SELECT BENE_ID, ICD_DGNS_CD1,            CLM_FROM_DT FROM outpatient
        ) dx ON d.BENE_ID = dx.BENE_ID
        WHERE dx.dx LIKE 'Z68%'
          AND CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
    )
    SELECT COUNT(DISTINCT n.BENE_ID) AS numerator
    FROM numerator n
""", conn)

,numerator
0,0


## Step 3 — Rate

In [4]:
pd.read_sql("""
    WITH age_eligible AS (
        SELECT BENE_ID
        FROM beneficiary
        WHERE BENE_DEATH_DT IS NULL
          AND FLOOR(DATEDIFF(day, CONVERT(date, BENE_BIRTH_DT, 106), '2021-12-31') / 365.25)
              BETWEEN 18 AND 74
    ),
    qualifying_visits AS (
        SELECT DISTINCT BENE_ID
        FROM (
            SELECT BENE_ID, HCPCS_CD, CLM_FROM_DT FROM carrier
            UNION ALL
            SELECT BENE_ID, HCPCS_CD, CLM_FROM_DT FROM outpatient
        ) h
        WHERE CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
          AND HCPCS_CD IN (
              '99385', '99386', '99387',
              '99395', '99396', '99397',
              '99202', '99203', '99204', '99205',
              '99212', '99213', '99214', '99215'
          )
    ),
    denominator AS (
        SELECT a.BENE_ID
        FROM age_eligible a
        INNER JOIN qualifying_visits v ON a.BENE_ID = v.BENE_ID
    ),
    numerator AS (
        SELECT DISTINCT d.BENE_ID
        FROM denominator d
        INNER JOIN (
            SELECT BENE_ID, LINE_ICD_DGNS_CD AS dx, CLM_FROM_DT FROM carrier
            UNION ALL
            SELECT BENE_ID, PRNCPAL_DGNS_CD,         CLM_FROM_DT FROM outpatient
            UNION ALL
            SELECT BENE_ID, ICD_DGNS_CD1,            CLM_FROM_DT FROM outpatient
        ) dx ON d.BENE_ID = dx.BENE_ID
        WHERE dx.dx LIKE 'Z68%'
          AND CONVERT(date, CLM_FROM_DT, 106) BETWEEN '2021-01-01' AND '2021-12-31'
    )
    SELECT
        COUNT(DISTINCT d.BENE_ID)                                        AS denominator,
        COUNT(DISTINCT n.BENE_ID)                                        AS numerator,
        ROUND(
            CAST(COUNT(DISTINCT n.BENE_ID) AS FLOAT)
            / NULLIF(COUNT(DISTINCT d.BENE_ID), 0) * 100, 1
        )                                                                AS rate_pct
    FROM denominator d
    LEFT JOIN numerator n ON d.BENE_ID = n.BENE_ID
""", conn)

,denominator,numerator,rate_pct
0,341,0,0.0


---
## Conclusion: Not implemented

The denominator is valid — 341 members with qualifying outpatient visits in 2021.
However, the numerator returns 0. ICD-10 Z68.x BMI documentation codes are absent
from the CMS synthetic FFS dataset. Synthea does not generate these codes.

This is the same pattern observed with BCS (mammography) and LBP (lumbar imaging) —
the denominator population exists but the specific numerator codes are not present
in the synthetic data. A 0% rate would not reflect real-world performance and is
not meaningful.

ABA will not be implemented in this project.